In [1]:
import polars as pl

In [ ]:
fact = pl.read_parquet("../data/gold/fact_trafico_hora.parquet")
fecha = pl.read_parquet("../data/gold/dim_fecha.parquet")
sensor = pl.read_parquet("../data/gold/dim_sensor.parquet")

In [3]:
print(f"Fact: {fact.shape}")
print(f"Fecha: {fecha.shape}")
print(f"Sensor: {sensor.shape}")

Fact: (40521393, 13)
Fecha: (365, 8)
Sensor: (5088, 9)


In [4]:
fact.schema

Schema([('id_sensor', Int32),
        ('id_fecha', Date),
        ('hora', Int32),
        ('intensidad_media', Float64),
        ('intensidad_max', Float64),
        ('intensidad_min', Float64),
        ('ocupacion_media', Float64),
        ('ocupacion_max', Float64),
        ('velocidad_media', Float64),
        ('velocidad_min', Float64),
        ('num_mediciones', Int64),
        ('num_error_E', Float64),
        ('porcentaje_calidad', Float64)])

In [5]:
fecha.schema

Schema([('id_fecha', Date),
        ('año', Int64),
        ('mes', Int64),
        ('nombre_mes', String),
        ('trimestre', Int64),
        ('dia', Int64),
        ('dia_semana', Int64),
        ('fin_semana', Boolean)])

In [6]:
sensor.schema

Schema([('id_sensor', Int32),
        ('tipo_elem', String),
        ('distrito', Int32),
        ('cod_cent', String),
        ('nombre_norm', String),
        ('utm_x', Float64),
        ('utm_y', Float64),
        ('latitud', Float64),
        ('longitud', Float64)])

In [7]:
fact.head()

id_sensor,id_fecha,hora,intensidad_media,intensidad_max,intensidad_min,ocupacion_media,ocupacion_max,velocidad_media,velocidad_min,num_mediciones,num_error_E,porcentaje_calidad
i32,date,i32,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64
1001,2026-04-18,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1001,2026-04-21,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1001,2026-04-21,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1002,2026-04-16,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1002,2026-04-17,21,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0


In [8]:
fecha.head()

id_fecha,año,mes,nombre_mes,trimestre,dia,dia_semana,fin_semana
date,i64,i64,str,i64,i64,i64,bool
2026-04-17,2026,4,"""April""",2,17,5,false
2026-04-21,2026,4,"""April""",2,21,2,false
2026-04-23,2026,4,"""April""",2,23,4,false
2026-04-25,2026,4,"""April""",2,25,6,true
2026-04-11,2026,4,"""April""",2,11,6,true


In [9]:
sensor.head()

id_sensor,tipo_elem,distrito,cod_cent,nombre_norm,utm_x,utm_y,latitud,longitud
i32,str,i32,str,str,f64,f64,f64,f64
1013,"""other""",9,"""18XC46PM01""","""18xc46pm01""",438704.110899,4.4746e6,40.419971,-3.722532
1019,"""other""",12,"""13XL49PM01""","""13xl49pm01""",440866.291144,4.4710e6,40.387345,-3.696709
1020,"""other""",2,"""13NL60PM01""","""13nl60pm01""",440751.95328,4.4712e6,40.389482,-3.698078
1022,"""other""",12,"""14RR28PM01""","""14rr28pm01""",440316.937491,4.4717e6,40.393637,-3.703247
1031,"""other""",2,"""15RR60PM01""","""15rr60pm01""",439559.313876,4.4724e6,40.40015,-3.712242


In [10]:
print(f"Sensores fact: {fact['id_sensor'].n_unique()}")
print(f"Sensores dimensión: {sensor['id_sensor'].n_unique()}")

Sensores fact: 4933
Sensores dimensión: 5088


In [11]:
sensores_faltantes = (
    set(fact["id_sensor"].unique())
    - set(sensor["id_sensor"].unique())
)

print(len(sensores_faltantes))

0


In [12]:
df = (
    fact
    .join(fecha, on="id_fecha", how="left")
    .join(sensor, on="id_sensor", how="left")
)

In [13]:
print(fact.height)
print(df.height)

40521393
40521393


In [14]:
df.select(
    pl.col("año").is_null().sum()
)

año
u32
0


In [15]:
df.select(
    pl.col("tipo_elem").is_null().sum()
)

tipo_elem
u32
0


In [16]:
clave_unica = df.select(
    pl.struct(
        ["id_sensor","id_fecha","hora"]
    ).n_unique()
).item()

print(f"Filas: {df.height}")
print(f"Claves únicas: {clave_unica}")

Filas: 40521393
Claves únicas: 40521393


In [17]:
(
    df.null_count()
      .transpose(
          include_header=True,
          header_name="columna",
          column_names=["nulos"]
      )
      .sort("nulos", descending=True)
)

columna,nulos
str,u32
"""nombre_norm""",111090
"""distrito""",42529
"""velocidad_media""",33781
"""velocidad_min""",33781
"""id_sensor""",0
…,…
"""cod_cent""",0
"""utm_x""",0
"""utm_y""",0


In [18]:
df = df.drop(
    "nombre_norm",
    "cod_cent",
    "utm_x",
    "utm_y"
)

In [19]:
df = df.with_columns(
    pl.col("distrito")
      .fill_null(0)
      .cast(pl.Int32)
)

In [20]:
df.group_by("tipo_elem").agg([
    pl.col("velocidad_media")
      .filter(pl.col("velocidad_media") > 0)
      .len()
      .alias("registros_con_velocidad")
])

tipo_elem,registros_con_velocidad
str,u32
"""other""",762297
"""M30""",2361374
"""URB""",1


In [21]:
# 1. Tratamiento inicial de la velocidad

df = df.with_columns([

    pl.when(pl.col("tipo_elem") == "URB")
      .then(0)
      .when(pl.col("velocidad_media") < 0)
      .then(None)
      .otherwise(pl.col("velocidad_media"))
      .alias("velocidad_media"),

    pl.when(pl.col("tipo_elem") == "URB")
      .then(0)
      .when(pl.col("velocidad_min") < 0)
      .then(None)
      .otherwise(pl.col("velocidad_min"))
      .alias("velocidad_min")

])

In [22]:
# 2. Calcular la mediana por sensor

medianas = (
    df.group_by("id_sensor")
      .agg([
          pl.col("velocidad_media").median().alias("mediana_media"),
          pl.col("velocidad_min").median().alias("mediana_min")
      ])
)

df = df.join(medianas, on="id_sensor", how="left")

In [23]:
# 3. Imputar los nulos con la mediana del sensor

df = df.with_columns([

    pl.col("velocidad_media")
      .fill_null(pl.col("mediana_media"))
      .alias("velocidad_media"),

    pl.col("velocidad_min")
      .fill_null(pl.col("mediana_min"))
      .alias("velocidad_min")

]).drop(["mediana_media", "mediana_min"])

In [24]:
df.select([
    pl.col("velocidad_media").is_null().sum(),
    pl.col("velocidad_min").is_null().sum()
])

velocidad_media,velocidad_min
u32,u32
0,0


In [25]:
(
    df.null_count()
      .transpose(
          include_header=True,
          header_name="columna",
          column_names=["nulos"]
      )
      .sort("nulos", descending=True)
)

columna,nulos
str,u32
"""id_sensor""",0
"""id_fecha""",0
"""hora""",0
"""intensidad_media""",0
"""intensidad_max""",0
…,…
"""fin_semana""",0
"""tipo_elem""",0
"""distrito""",0


In [26]:
# Comprobar si existen registros con el valor 99999

df.filter(
    pl.col("intensidad_max") == 99999
).select(pl.len())

len
u32
1


In [27]:
# Inspeccionar el registro antes de eliminarlo

df.filter(
    pl.col("intensidad_max") == 99999
).select([
    "id_sensor",
    "id_fecha",
    "hora",
    "intensidad_media",
    "intensidad_max",
    "intensidad_min",
    "num_mediciones"
])

id_sensor,id_fecha,hora,intensidad_media,intensidad_max,intensidad_min,num_mediciones
i32,date,i32,f64,f64,f64,i64
7112,2025-12-12,2,50083.0,99999.0,167.0,2


In [28]:
# Analizar las intensidades más elevadas del conjunto de datos

df.filter(
    pl.col("intensidad_media") > 10000
).select([
    "id_sensor",
    "id_fecha",
    "hora",
    "intensidad_media",
    "intensidad_max",
    "intensidad_min",
    "num_mediciones"
]).sort(
    "intensidad_media",
    descending=True
)

id_sensor,id_fecha,hora,intensidad_media,intensidad_max,intensidad_min,num_mediciones
i32,date,i32,f64,f64,f64,i64
7023,2026-02-20,22,91368.333333,91544.0,91034.0,3
7023,2026-02-17,13,91338.25,91560.0,90707.0,4
7023,2026-02-20,23,91269.25,91517.0,90560.0,4
7023,2026-02-17,3,91191.5,91445.0,90932.0,4
7023,2026-02-21,7,91068.0,91500.0,90636.0,2
…,…,…,…,…,…,…
5670,2025-08-20,17,10283.0,22626.0,4889.0,4
5670,2025-07-16,10,10038.25,11225.0,8136.0,4
5670,2026-03-18,15,10033.75,13213.0,4523.0,4


In [29]:
# Eliminar únicamente el registro con el valor centinela 99999

df = df.filter(
    pl.col("intensidad_max") != 99999
)

In [ ]:
df.write_parquet("../data/modeling/base_modelado.parquet")